Find first non null phone per user(Filter nulls rank by timestamp)

In [0]:
-- Create table: user phone records (some phones are NULL)
CREATE OR REPLACE TEMPORARY TABLE user_phones (
  user_id   INT,
  phone_number     STRING,
  updated_at        TIMESTAMP
);

-- Insert 5 records (includes NULL phones and multiple rows per user)
INSERT INTO user_phones VALUES
  (1, NULL,             '2024-01-01 10:00:00'),
  (1, '9876543210',     '2024-01-03 10:00:00'),
  (2, '5551234567',     '2024-01-02 09:30:00'),
  (2, NULL,             '2024-01-04 11:00:00'),
  (3, '9998887776',     '2024-01-05 08:15:00');

SELECT * FROM user_phones ORDER BY user_id, updated_at;

-- Find the first non-null phone per user:
-- filter out NULL phones, rank by timestamp (earliest first), pick rank = 1
WITH ranked AS (
  SELECT
    user_id,
    phone_number,
    updated_at,
    ROW_NUMBER() OVER (
      PARTITION BY user_id
      ORDER BY updated_at
    ) AS rn
  FROM user_phones
  WHERE phone_number IS NOT NULL
)
SELECT user_id, phone_number AS first_non_null_phone, updated_at
FROM ranked
WHERE rn = 1
ORDER BY user_id;

In [0]:
WITH filtered AS (
  SELECT
    user_id,
    phone_number,
    updated_at,
    ROW_NUMBER() OVER (
      PARTITION BY user_id
      ORDER BY updated_at
    ) AS rn
  FROM user_phones
  WHERE phone_number IS NOT NULL
)
SELECT
  user_id,
  phone_number AS first_non_null_phone
FROM filtered
WHERE rn = 1
ORDER BY user_id;

In [0]:
-- Sample sales data (order_id, amount, order_date)
CREATE OR REPLACE TEMPORARY TABLE sales_orders (
  order_id INT,
  amount DECIMAL(10,2),
  order_date DATE
);

INSERT INTO sales_orders VALUES
  (101, 500.00, '2024-01-05'),
  (102, 300.00, '2024-01-20'),
  (103, 700.00, '2024-02-03'),
  (104, 200.00, '2024-02-15'),
  (105, 900.00, '2024-03-10'),
  (106, 450.00, '2024-04-08'),
  (107, 650.00, '2024-04-22'),
  (108, 800.00, '2024-05-01');

-- Monthly revenue + running total:
-- aggregate revenue by month first, then use a window SUM
-- ordered by month to get the cumulative total


In [0]:
WITH monthly_revenue AS (
  SELECT
    date_format(order_date, 'yyyy-MM') AS revenue_month,
    SUM(amount)                      AS monthly_revenue
  FROM sales_orders
  GROUP BY revenue_month
)
SELECT
  revenue_month,
  monthly_revenue,
  SUM(monthly_revenue) OVER (
    ORDER BY revenue_month
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
  ) AS running_total
FROM monthly_revenue
ORDER BY revenue_month;